In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# Transforms
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3803, 0.4078],
                         std=[0.2038, 0.1367, 0.1147])
])

val_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3803, 0.4078],
                         std=[0.2038, 0.1367, 0.1147])
])

# Datasets
train_data = datasets.EuroSAT(root="./data", transform=train_transforms, download=True)
val_data   = datasets.EuroSAT(root="./data", transform=val_transforms,   download=False)

# Split
train_dataset, val_dataset = random_split(train_data, [21600, 5400],
                                          generator=torch.Generator().manual_seed(42))
val_dataset.dataset = val_data

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

# Training functions
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to('cuda'), labels.to('cuda')
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to('cuda'), labels.to('cuda')
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

print("✅ Everything ready")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Device: {torch.cuda.get_device_name(0)}")

100%|██████████| 94.3M/94.3M [00:00<00:00, 287MB/s]


✅ Everything ready
Train: 21600 | Val: 5400
Device: Tesla T4


In [2]:
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)
model = model.to('cuda')

# See ResNet18 layer structure
for name, param in model.named_parameters():
    print(f"{name:45s} | trainable: {param.requires_grad}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 102MB/s] 


conv1.weight                                  | trainable: True
bn1.weight                                    | trainable: True
bn1.bias                                      | trainable: True
layer1.0.conv1.weight                         | trainable: True
layer1.0.bn1.weight                           | trainable: True
layer1.0.bn1.bias                             | trainable: True
layer1.0.conv2.weight                         | trainable: True
layer1.0.bn2.weight                           | trainable: True
layer1.0.bn2.bias                             | trainable: True
layer1.1.conv1.weight                         | trainable: True
layer1.1.bn1.weight                           | trainable: True
layer1.1.bn1.bias                             | trainable: True
layer1.1.conv2.weight                         | trainable: True
layer1.1.bn2.weight                           | trainable: True
layer1.1.bn2.bias                             | trainable: True
layer2.0.conv1.weight                   

In [3]:
# Start fresh
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)

# Step 1 — freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2 — unfreeze layer4 + fc only
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')

# Verify
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,}")

# Use lower LR for fine-tuning — critical
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)  # 10x lower than before

history_ft = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   []
}

print(f"\n{'Epoch':>5} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 32)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)

    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['val_loss'].append(val_loss)
    history_ft['val_acc'].append(val_acc)

    print(f"  {epoch+1:02d}   |   {train_acc:.4f}  |  {val_acc:.4f}")

print(f"\n=== Results ===")
print(f"Frozen backbone:     0.8530")
print(f"layer4 fine-tuned:   {history_ft['val_acc'][-1]:.4f}")

Trainable: 8,398,858 / 11,181,642

Epoch | Train Acc |  Val Acc
--------------------------------


Validation: 100%|██████████| 169/169 [00:03<00:00, 43.72it/s]


  01   |   0.8166  |  0.9157


Validation: 100%|██████████| 169/169 [00:03<00:00, 53.40it/s]


  02   |   0.8775  |  0.9328


...

  08   |   0.9235  |  0.9502


Validation: 100%|██████████| 169/169 [00:02<00:00, 56.73it/s]


  09   |   0.9267  |  0.9439


Validation: 100%|██████████| 169/169 [00:04<00:00, 40.82it/s]

  10   |   0.9286  |  0.9444

=== Results ===
Frozen backbone:     0.8530
layer4 fine-tuned:   0.9444


In [4]:
# Unfreeze entire network
for param in model.parameters():
    param.requires_grad = True

# Even lower LR — touching early layers is risky
optimizer = optim.Adam(model.parameters(), lr=0.00001)

print(f"{'Epoch':>5} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 32)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)

    history_ft['train_loss'].append(train_loss)
    history_ft['train_acc'].append(train_acc)
    history_ft['val_loss'].append(val_loss)
    history_ft['val_acc'].append(val_acc)

    print(f"  {epoch+11:02d}   |   {train_acc:.4f}  |  {val_acc:.4f}")

print(f"\n=== Full Comparison ===")
print(f"Frozen backbone:     0.8530")
print(f"layer4 fine-tuned:   0.9404")
print(f"Full fine-tuned:     {history_ft['val_acc'][-1]:.4f}")

Epoch | Train Acc |  Val Acc
--------------------------------


Validation: 100%|██████████| 169/169 [00:03<00:00, 52.62it/s]


  11   |   0.9439  |  0.9535


Validation: 100%|██████████| 169/169 [00:03<00:00, 56.07it/s]


  12   |   0.9471  |  0.9576


...

  18   |   0.9633  |  0.9630


Validation: 100%|██████████| 169/169 [00:03<00:00, 51.11it/s]


  19   |   0.9648  |  0.9661


Validation: 100%|██████████| 169/169 [00:03<00:00, 53.60it/s]

  20   |   0.9651  |  0.9578

=== Full Comparison ===
Frozen backbone:     0.8530
layer4 fine-tuned:   0.9404
Full fine-tuned:     0.9578


In [5]:
def run_experiment(optimizer_name, lr, epochs=5):
    model = models.resnet18(weights='IMAGENET1K_V1')
    model.fc = nn.Linear(512, 10)
    
    # Unfreeze layer4 + fc — same config as best result
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    
    model = model.to('cuda')
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    print(f"{optimizer_name.upper():>5} | LR: {lr:.0e} | Val Acc: {val_acc:.4f}")
    return val_acc

# Compare
print("=== SGD vs Adam ===")
run_experiment('adam', lr=0.0001)
run_experiment('sgd',  lr=0.001)
run_experiment('sgd',  lr=0.01)

=== SGD vs Adam ===


Validation: 100%|██████████| 169/169 [00:03<00:00, 48.74it/s]


 ADAM | LR: 1e-04 | Val Acc: 0.9385


Validation: 100%|██████████| 169/169 [00:03<00:00, 53.61it/s]


  SGD | LR: 1e-03 | Val Acc: 0.9346


Validation: 100%|██████████| 169/169 [00:03<00:00, 43.60it/s]

  SGD | LR: 1e-02 | Val Acc: 0.9446


0.9446296296296296

In [6]:
model = models.resnet18(weights='IMAGENET1K_V1')
model.fc = nn.Linear(512, 10)

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Reduce LR by 0.1 every 5 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print(f"{'Epoch':>5} | {'LR':>8} | {'Train Acc':>9} | {'Val Acc':>8}")
print("-" * 45)

for epoch in range(10):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc     = val_epoch(model, val_loader, criterion)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  {epoch+1:02d}   | {current_lr:.0e} |   {train_acc:.4f}  |  {val_acc:.4f}")
    
    # Step scheduler after each epoch
    scheduler.step()

Epoch |       LR | Train Acc |  Val Acc
---------------------------------------------


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.68it/s]


  01   | 1e-04 |   0.8147  |  0.9169


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.51it/s]


  02   | 1e-04 |   0.8798  |  0.9320


...

  08   | 1e-05 |   0.9276  |  0.9470


Validation: 100%|██████████| 169/169 [00:03<00:00, 42.36it/s]


  09   | 1e-05 |   0.9329  |  0.9446


Validation: 100%|██████████| 169/169 [00:03<00:00, 50.00it/s]

  10   | 1e-05 |   0.9306  |  0.9452


In [7]:
# Custom ResNet with Dropout before final layer
class ResNetWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.base = models.resnet18(weights='IMAGENET1K_V1')
        
        # Replace fc with dropout + linear
        self.base.fc = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.base(x)

# Test three dropout rates
for rate in [0.0, 0.3, 0.5]:
    model = ResNetWithDropout(dropout_rate=rate)
    
    # Unfreeze layer4 + fc
    for param in model.parameters():
        param.requires_grad = False
    for param in model.base.layer4.parameters():
        param.requires_grad = True
    for param in model.base.fc.parameters():
        param.requires_grad = True

    model = model.to('cuda')
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    print(f"Dropout: {rate} | Val Acc: {val_acc:.4f}")

Validation: 100%|██████████| 169/169 [00:03<00:00, 54.54it/s]


Dropout: 0.0 | Val Acc: 0.9469


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.54it/s]


Dropout: 0.3 | Val Acc: 0.9380


Validation: 100%|██████████| 169/169 [00:03<00:00, 48.67it/s]

Dropout: 0.5 | Val Acc: 0.9393


In [8]:
# Compare weight decay values
for wd in [0, 1e-4, 1e-3, 1e-2]:
    model = ResNetWithDropout(dropout_rate=0.3)

    for param in model.parameters():
        param.requires_grad = False
    for param in model.base.layer4.parameters():
        param.requires_grad = True
    for param in model.base.fc.parameters():
        param.requires_grad = True

    model = model.to('cuda')
    criterion = nn.CrossEntropyLoss()
    
    # Weight decay is built into optimizer
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=wd)

    for epoch in range(5):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    print(f"Weight Decay: {wd:.0e} | Val Acc: {val_acc:.4f}")

Validation: 100%|██████████| 169/169 [00:03<00:00, 42.89it/s]


Weight Decay: 0e+00 | Val Acc: 0.9328


Validation: 100%|██████████| 169/169 [00:03<00:00, 55.60it/s]


Weight Decay: 1e-04 | Val Acc: 0.9402


Validation: 100%|██████████| 169/169 [00:04<00:00, 40.06it/s]


Weight Decay: 1e-03 | Val Acc: 0.9454


Validation: 100%|██████████| 169/169 [00:03<00:00, 52.62it/s]

Weight Decay: 1e-02 | Val Acc: 0.9359


In [9]:
import os

def train_with_checkpointing(model, train_loader, val_loader, 
                              criterion, optimizer, epochs, 
                              save_path='best_model.pth'):
    best_val_acc = 0.0
    best_epoch   = 0

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc     = val_epoch(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch   = epoch + 1
            torch.save({
                'epoch':      epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc':    val_acc,
                'val_loss':   val_loss,
            }, save_path)
            print(f"Epoch {epoch+1:02d} | Val Acc: {val_acc:.4f} ← saved best ✅")
        else:
            print(f"Epoch {epoch+1:02d} | Val Acc: {val_acc:.4f}")

    print(f"\nBest model: epoch {best_epoch} | Val Acc: {best_val_acc:.4f}")
    return history

# Build model
model = ResNetWithDropout(dropout_rate=0.3)
for param in model.parameters():
    param.requires_grad = False
for param in model.base.layer4.parameters():
    param.requires_grad = True
for param in model.base.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

history = train_with_checkpointing(
    model, train_loader, val_loader,
    criterion, optimizer, epochs=10
)

Validation: 100%|██████████| 169/169 [00:03<00:00, 45.95it/s]


Epoch 01 | Val Acc: 0.9152 ← saved best ✅


Validation: 100%|██████████| 169/169 [00:03<00:00, 45.88it/s]


Epoch 02 | Val Acc: 0.9357 ← saved best ✅


Validation: 100%|██████████| 169/169 [00:03<00:00, 54.14it/s]


...

Epoch 08 | Val Acc: 0.9444


Validation: 100%|██████████| 169/169 [00:03<00:00, 56.28it/s]


Epoch 09 | Val Acc: 0.9370


Validation: 100%|██████████| 169/169 [00:03<00:00, 50.83it/s]

Epoch 10 | Val Acc: 0.9467

Best model: epoch 6 | Val Acc: 0.9483


In [10]:
def train_with_early_stopping(model, train_loader, val_loader,
                               criterion, optimizer, epochs,
                               patience=3, save_path='best_model.pth'):
    best_val_acc  = 0.0
    best_epoch    = 0
    epochs_no_improve = 0

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc     = val_epoch(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc      = val_acc
            best_epoch        = epoch + 1
            epochs_no_improve = 0
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc':              val_acc,
            }, save_path)
            print(f"Epoch {epoch+1:02d} | Val Acc: {val_acc:.4f} ← saved best ✅")
        else:
            epochs_no_improve += 1
            print(f"Epoch {epoch+1:02d} | Val Acc: {val_acc:.4f} | No improve: {epochs_no_improve}/{patience}")

            if epochs_no_improve >= patience:
                print(f"\n⛔ Early stopping at epoch {epoch+1}")
                break

    print(f"\nBest model: epoch {best_epoch} | Val Acc: {best_val_acc:.4f}")
    return history

# Fresh model
model = ResNetWithDropout(dropout_rate=0.3)
for param in model.parameters():
    param.requires_grad = False
for param in model.base.layer4.parameters():
    param.requires_grad = True
for param in model.base.fc.parameters():
    param.requires_grad = True

model = model.to('cuda')
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

history = train_with_early_stopping(
    model, train_loader, val_loader,
    criterion, optimizer,
    epochs=50,          # set high — early stopping will cut it short
    patience=3
)

Validation: 100%|██████████| 169/169 [00:03<00:00, 50.87it/s]


Epoch 01 | Val Acc: 0.9157 ← saved best ✅


Validation: 100%|██████████| 169/169 [00:03<00:00, 45.02it/s]


Epoch 02 | Val Acc: 0.9300 ← saved best ✅


Validation: 100%|██████████| 169/169 [00:03<00:00, 54.14it/s]


...

Epoch 07 | Val Acc: 0.9376 | No improve: 1/3


Validation: 100%|██████████| 169/169 [00:04<00:00, 41.27it/s]


Epoch 08 | Val Acc: 0.9437 | No improve: 2/3


Validation: 100%|██████████| 169/169 [00:03<00:00, 53.93it/s]

Epoch 09 | Val Acc: 0.9426 | No improve: 3/3

⛔ Early stopping at epoch 9

Best model: epoch 6 | Val Acc: 0.9459


In [11]:
 # Load best model from disk
checkpoint = torch.load('best_model.pth')

# Rebuild model architecture
model_loaded = ResNetWithDropout(dropout_rate=0.3)
model_loaded.load_state_dict(checkpoint['model_state_dict'])
model_loaded = model_loaded.to('cuda')

# Verify it matches saved accuracy
val_loss, val_acc = val_epoch(model_loaded, val_loader, criterion)
print(f"Saved at epoch:     {checkpoint['epoch'] + 1}")
print(f"Saved val acc:      {checkpoint['val_acc']:.4f}")
print(f"Reloaded val acc:   {val_acc:.4f}")
print(f"Match: {abs(val_acc - checkpoint['val_acc']) < 0.001}")

Validation: 100%|██████████| 169/169 [00:03<00:00, 43.57it/s]

Saved at epoch:     6
Saved val acc:      0.9459
Reloaded val acc:   0.9459
Match: True
